# Nettoyage des données de test

Ce notebook applique exactement les mêmes transformations que dans `Train.ipynb` sur le dataset de test `wind_turbine_maintenance_test_data.csv`.

Le fichier nettoyé est sauvegardé dans `data/processed/` pour être utilisé dans le notebook de prédiction.

In [ ]:
import pandas as pd
import numpy as np
import boto3
import os
from pathlib import Path
from dotenv import load_dotenv

# Chargement des credentials AWS
load_dotenv()  # lit .env dans le dossier courant (notebook/)

S3_BUCKET    = "windscan"
S3_REGION    = "eu-north-1"
S3_DATA_KEY  = "data/processed/"

def upload_to_s3(local_path, s3_key):
    s3 = boto3.client(
        "s3", region_name=S3_REGION,
        aws_access_key_id=os.getenv("AWS_ACCESS_KEY_ID"),
        aws_secret_access_key=os.getenv("AWS_SECRET_ACCESS_KEY"),
    )
    s3.upload_file(str(local_path), S3_BUCKET, s3_key)
    print(f"Upload S3 OK : s3://{S3_BUCKET}/{s3_key}")

print(f"AWS key chargée : {bool(os.getenv('AWS_ACCESS_KEY_ID'))}")

## Chargement du dataset de test

In [20]:
dataset = pd.read_csv("../data/raw/wind_turbine_maintenance_test_data.csv")

print(f"Dimensions : {dataset.shape}")
print(f"Colonnes   : {list(dataset.columns)}")
display(dataset.head())

print("\nValeurs manquantes (%) :")
display(round(100 * dataset.isnull().sum() / len(dataset), 2))

Dimensions : (35040, 10)
Colonnes   : ['Turbine_ID', 'Rotor_Speed_RPM', 'Wind_Speed_mps', 'Power_Output_kW', 'Gearbox_Oil_Temp_C', 'Generator_Bearing_Temp_C', 'Vibration_Level_mmps', 'Ambient_Temp_C', 'Humidity_pct', 'Maintenance_Label']


,Turbine_ID,Rotor_Speed_RPM,Wind_Speed_mps,Power_Output_kW,Gearbox_Oil_Temp_C,Generator_Bearing_Temp_C,Vibration_Level_mmps,Ambient_Temp_C,Humidity_pct,Maintenance_Label
0,1,16.329212,7.229967,1468.371964,60.045948,69.645919,1.712257,12.822084,62.957219,2
1,1,14.109988,8.943831,1793.460409,58.544454,74.709346,2.117721,13.991153,40.500486,0
2,1,16.259475,9.562434,1492.022307,57.223514,77.115291,2.075517,18.011540,62.155541,0
3,1,13.420109,6.911183,1514.954999,57.041996,80.484269,1.849384,14.383540,48.726572,0
4,1,16.428984,5.910646,1487.018006,68.157615,72.067310,2.058144,16.320517,62.900348,0



Valeurs manquantes (%) :


Turbine_ID                  0.0
Rotor_Speed_RPM             0.0
Wind_Speed_mps              0.0
Power_Output_kW             0.0
Gearbox_Oil_Temp_C          0.0
Generator_Bearing_Temp_C    0.0
Vibration_Level_mmps        0.0
Ambient_Temp_C              0.0
Humidity_pct                0.0
Maintenance_Label           0.0
dtype: float64

## Séparation par turbine

On conserve uniquement la **Turbine 1**, cohérent avec le modèle entraîné sur cette turbine.

In [21]:
print(f"Distribution Turbine_ID :\n{dataset['Turbine_ID'].value_counts()}")

Turbine1_test = dataset[dataset["Turbine_ID"] == 1].copy()
print(f"\nTurbine 1 — {len(Turbine1_test)} lignes")

Distribution Turbine_ID :
Turbine_ID
1    17520
2    17520
Name: count, dtype: int64

Turbine 1 — 17520 lignes


## Nettoyage

Suppression des mêmes colonnes que dans `Train.ipynb` : `Turbine_ID`, `Ambient_Temp_C`, `Humidity_pct`.

In [22]:
useless_cols = ["Turbine_ID", "Ambient_Temp_C", "Humidity_pct"]

Turbine1_test = Turbine1_test.drop(useless_cols, axis=1)

print(f"Colonnes restantes : {list(Turbine1_test.columns)}")
print(f"Dimensions finales : {Turbine1_test.shape}")
display(Turbine1_test.head())

Colonnes restantes : ['Rotor_Speed_RPM', 'Wind_Speed_mps', 'Power_Output_kW', 'Gearbox_Oil_Temp_C', 'Generator_Bearing_Temp_C', 'Vibration_Level_mmps', 'Maintenance_Label']
Dimensions finales : (17520, 7)


,Rotor_Speed_RPM,Wind_Speed_mps,Power_Output_kW,Gearbox_Oil_Temp_C,Generator_Bearing_Temp_C,Vibration_Level_mmps,Maintenance_Label
0,16.329212,7.229967,1468.371964,60.045948,69.645919,1.712257,2
1,14.109988,8.943831,1793.460409,58.544454,74.709346,2.117721,0
2,16.259475,9.562434,1492.022307,57.223514,77.115291,2.075517,0
3,13.420109,6.911183,1514.954999,57.041996,80.484269,1.849384,0
4,16.428984,5.910646,1487.018006,68.157615,72.067310,2.058144,0


## Vérification de la cohérence

On vérifie que les colonnes correspondent bien à celles vues à l'entraînement.

In [23]:
# Colonnes attendues apres nettoyage (identiques au train set)
expected_cols = [
    "Rotor_Speed_RPM", "Wind_Speed_mps", "Power_Output_kW",
    "Gearbox_Oil_Temp_C", "Generator_Bearing_Temp_C",
    "Vibration_Level_mmps", "Maintenance_Label"
]

missing = [c for c in expected_cols if c not in Turbine1_test.columns]
extra   = [c for c in Turbine1_test.columns if c not in expected_cols]

if not missing and not extra:
    print("Colonnes conformes au train set.")
else:
    if missing: print(f"Colonnes manquantes : {missing}")
    if extra:   print(f"Colonnes en trop    : {extra}")

Colonnes conformes au train set.


## Sauvegarde

In [24]:
output_path = "../data/processed/wind_turbine_maintenance_test_data_cleaned.csv"
Turbine1_test.to_csv(output_path, index=False)
print(f"Dataset nettoyé sauvegardé : {output_path}")
print(f"Dimensions : {Turbine1_test.shape}")

Dataset nettoyé sauvegardé : ../data/processed/wind_turbine_maintenance_test_data_cleaned.csv
Dimensions : (17520, 7)


In [ ]:
# Upload S3
output_file = Path(output_path).resolve()
s3_key = S3_DATA_KEY + Path(output_path).name
upload_to_s3(output_file, s3_key)